In [146]:
import pandas as pd
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from utils import get_all_jpg_pics,edge_extraction,detect_shapes,compute_gabor_features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

%matplotlib inline

In [147]:
IMGAE_ROOT = "data/Image/"
DATA_ROOT = "data/Scores/"

final_product_images = get_all_jpg_pics(IMGAE_ROOT+"Final-Product/")
timeseries_images = get_all_jpg_pics(IMGAE_ROOT+"Time-Based/")



In [148]:
all_images = final_product_images + timeseries_images
df_all_scores = pd.read_csv(DATA_ROOT+"Unsupervised_CV_scores.csv")

In [149]:
df_all_scores

,Sample name,Score_unsup_CV,Score,TS,texture_unsup_CV,texture_real
0,901 h1,8.646532,8,901,0,0
1,901 h2,8.627842,8,901,0,0
2,901 v1,8.707538,5,901,0,1
3,901 v2,8.706868,6,901,0,0
4,902 h1,8.605621,7,902,0,0
...,...,...,...,...,...,...
76,909 - 0,6.425028,7,909,0,0
77,909 - 2,5.178385,8,909,1,0
78,909 - 4,6.160581,9,909,0,0
79,909 - 6,4.816000,7,909,1,0


In [150]:
texture_features_list = []
edge_features_list = []
real_score_list = []
unsup_cv_score_list = []
image_names_list = []

edge_x = 60
edge_y = 80
for idx, pic in enumerate(all_images):
    print(f"Processing image {idx+1} of {len(all_images)}", end='\r')
    image_name = os.path.splitext(os.path.basename(pic))[0]
    real_score = df_all_scores.loc[df_all_scores['Sample name'] == image_name, 'Score'].values[0]
    cv_score = df_all_scores.loc[df_all_scores['Sample name'] == image_name, 'Score_unsup_CV'].values[0]
    
    img = cv2.imread(pic)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (0, 0), fx=0.2, fy=0.2)
    
    _,edge = edge_extraction(img, edge_x, edge_y)
    
    prosity, fibour = detect_shapes(edge)
    total = prosity + fibour
    porosity_score_edge = prosity / total
    fiber_score_edge = fibour / total 
    cv_score = 10 - (porosity_score_edge * 9)

    
    texture_feature = compute_gabor_features(img)
    
    image_names_list.append(image_name)
    texture_features_list.append(texture_feature)
    edge_features_list.append([porosity_score_edge, fiber_score_edge])
    unsup_cv_score_list.append(cv_score)
    real_score_list.append(real_score)

df_features = pd.DataFrame({
    'Sample Name': image_names_list,
    'Texture Feature': texture_features_list,
    'Edge Features 1': [e[0] for e in edge_features_list],
    'Edge Features 2': [e[1] for e in edge_features_list],
    'Unsup CV Score': unsup_cv_score_list,
    'Score': real_score_list
})


        

In [151]:
texture_features_array = np.array(texture_features_list)
edge_features_array = np.array(edge_features_list)
real_score_array = np.array(real_score_list)
unsup_cv_score_array = np.array(unsup_cv_score_list)

In [152]:
input_features = ['Texture Feature', 'Unsup CV Score', 'Edge Features 1', 'Edge Features 2']
target_feature = ['Score']

df_train, df_test = train_test_split(df_features, test_size=0.3,shuffle=True,random_state=42)
df_train['Split'] = 'Train'
df_test['Split'] = 'Test'
X_train = df_train[input_features].values
y_train = df_train[target_feature].values.ravel()
X_test = df_test[input_features].values
y_test = df_test[target_feature].values.ravel()

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
xgb_model = XGBRegressor(n_estimators=5,random_state=42,objective='reg:squarederror')
xgb_model.fit(X_train_scaled, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=5, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [153]:
df_train_pred = df_train.copy()
df_test_pred = df_test.copy()

y_pred_train = xgb_model.predict(X_train_scaled)
df_train_pred['Score_semi_sup_CV'] = y_pred_train
y_pred_test = xgb_model.predict(X_test_scaled)
df_test_pred['Score_semi_sup_CV'] = y_pred_test


In [154]:
df_test_pred.columns
mae = mean_absolute_error(y_test, y_pred_test)
print(f"Test MAE: {mae:.2f}")

Test MAE: 1.09


In [155]:
df_semisup_cv_scores = pd.concat([df_train_pred,df_test_pred],ignore_index=True)


In [156]:
df_semisup_cv_scores_evals = df_semisup_cv_scores[['Sample Name','Score','Score_semi_sup_CV','Split']]
df_semisup_cv_scores_evals['TS'] = df_semisup_cv_scores_evals['Sample Name'].str[:3]


C:\Users\amirbg\AppData\Local\Temp\ipykernel_1344\226955188.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_semisup_cv_scores_evals['TS'] = df_semisup_cv_scores_evals['Sample Name'].str[:3]


In [157]:
df_semisup_cv_scores_evals['Score_unsup_CV'] = df_semisup_cv_scores_evals['Sample Name'].map(
    df_all_scores.set_index('Sample name')['Score_unsup_CV']
)
df_semisup_cv_scores_evals.to_csv(DATA_ROOT+"All_CV_scores_evals.csv", index=False)

C:\Users\amirbg\AppData\Local\Temp\ipykernel_1344\1208971908.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_semisup_cv_scores_evals['Score_unsup_CV'] = df_semisup_cv_scores_evals['Sample Name'].map(


In [158]:
X = df_features[input_features].values
y= df_features[target_feature].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = XGBRegressor(n_estimators=5,random_state=42,objective='reg:squarederror')

model.fit(X_scaled,y.reshape(81,))

y_pred_all = model.predict(X_scaled)

In [159]:
df_all_scores['Score_semi_sup_CV'] = y_pred_all

In [160]:
df_all_scores.to_csv(DATA_ROOT+"All_CV_scores.csv",index=False)